<a href="https://colab.research.google.com/github/jacquemainainsley/TransitMethodPhotonCount/blob/main/Photon_Count_New_(Blackbody_Radiation).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Background Information
*   Task: Make a universal code to find spectra for the number of photons emitted from an exoplanet at the feature wavelengths of certain biosignature gases by **querying data from the NASA Exoplanet Archive** and **using the blackbody radiation equation**.
*   Given Information
    *   Spectral radiance (blackbody radiation) equation in terms of wavelength
        *   $B_{λ} = \frac{2hc^2}{λ^5} \times \frac{1}{e^\frac{hc}{λkt}}$
        *   Units: $\frac{W}{m^3 \times sr}$
    *   Energy of a photon equation
        *   $E_{photon} = \frac{hc}{λ}$
        *   Units: $J$
    *   The scaling factor for the calculation          
        *   $A \times t \times (\frac{R_{star}}{d})^2$
            *   $A$ = telescope collecting area ($cm^2$)
            *   $t$ = exposure time ($s$)
            *   $R_{star}$ = radius of the star ($m$)
            *   $d$ = distance from the star to the detector ($m$)
            *   Units: $cm^2 \times s$

*   Use $B_{λ}$, $E_{photon}$, and $A \times t \times (\frac{R_{star}}{d})^2$ to find $N$, the number of photons emitted from an exoplanet at an interval of wavelengths.
    *   $N = \int_{λ}^{λ} (\frac{2c}{λ^4} \times \frac{1}{e^\frac{hc}{λkt}} \times A \times t \times (\frac{R_{star}}{d})^2)dλ$
*   Stellar radius, effecive temperature, and distance will be queried from the NASA Exoplanet Archive for the a selected exoplanet.
    *   We will also assume that $A =$ 7.0685 $\times 10^6 cm^2$ and $t =$ 100 $s$.
*   Each biosignature gas uses its own feature wavelength ($λ$) range. This is the interval of the integral.
    *   $O_2 =$ 1.25 - 1.28 $μm$
    *   $CO_2 =$ 4.17 - 4.51 $μm$
    *   $CH_4 =$ 3.41 - 3.71 $μm$
    *   $H_2O =$ 1.76 - 1.94 $μm$
*   From here, we can evaluate the number of photons emitted for each biosignature gas at a feature wavelength range from an exoplanet. We can use this information, along with our calculated minimum photon count for a certain small error on our Poisson distribution, to decide how long to collect data for ($t$, exposure time) or how wide our detector should be ($A$, telescope collecting area) for a given exoplanet.



# Exoplanet Selection
*   We restricted the available exoplanets to those with the most appropriate conditions. In the search for exoplanets that are potentially habitable, we selected Earthlike candidates. We also selected exoplanets that are close and hosted by bright stars in order to maximize photon collection.
*   Conditions
    *   Observed by transit method
    *   Can sustain liquid water ($T_{eq} = [175 K, 647 K]$)
    *   Earthlike size $(R_{planet} =[0.7 R_E, 1.6 R_E]$)
    *   Close to observer ($d = [0 pc, 200 pc]$)
    *   Bright host star (KS-Band Magnitude = [0,8])
          * The KS-Band, an infrared band, is most appropriate for exoplanet observation.

*   Candidate Planets
    *   GJ 341 b
    *   HD 260655 b
    *   HD 260655 c
    *   GJ 357 b
    *   LTT 1445 A b
    *   L 98-59 b
    *   LHS 475 b
    *   Gliese 12 b
    *   GJ 3929 b
    *   TOI-244 b

*   Candidate Planet Extras
    *   LTT 1445 A c
    *   TOI-260 b
    *   L 98-59 c
    *   L 98-59 d
    *   HD 23472 f
    *   Kepler-409 b
    *   TOI 198 b
    *   Kepler-37 c
    *   LHS 1815 b

In [ ]:
# Select a planet by typing it in.
planet = "LTT 1445 A c"

In [ ]:
import numpy as np # Import numpy package, which allows us to conduct mathematical operations for the defined quantities.
import astropy.units as u # Import a Python package to call certain units.
import astropy.constants as const # Import a Python package to call the value of certain constants.

In [ ]:
# Define constants.
h = const.h # This is Planck's constant. (J/s)
c = const.c # This is the speed of light. (m/s)
k_B = const.k_B # This is Boltzmann's constant. (J/K)
pi = np.pi # This is pi.

In [ ]:
!pip install astroquery #Install the astroquery package in order to query astronomical data.
"""
Our notable service is the NASA Exoplanet Archive.
Although it does not follow a common and consistent API, it can be accessed through astroquery.ipac.nexsci.nasa_exoplanet_archive.
"""

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.0 MB/s eta 0:00:00


'\nOur notable service is the NASA Exoplanet Archive.\nAlthough it does not follow a common and consistent API, it can be accessed through astroquery.ipac.nexsci.nasa_exoplanet_archive.\n'

In [ ]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive # query data from the NASA Exoplanet Archive using astroquery.

# Define a function that extracts data for the chosen planets and files it into defined parameters.
def get_planet_data(planet_name):
    # 1. Define the extracted columns.
    select_columns_str = "pl_name,hostname,st_rad,sy_dist,st_teff,default_flag"

    # 2. Query the data for a specifically chosen exoplanet.
    planet_data = NasaExoplanetArchive.query_object(planet, table="ps", select=select_columns_str)

    # 3. Filter for the default (publiished confirmed) row.
    default_data = planet_data[planet_data['default_flag'] == 1]

    # 4. Extract specific parameters (using .value to get floats instead of masked quantities).
    T = default_data['st_teff'][0] # Extract the stellar effective temperature.
    R_star = default_data['st_rad'][0] # Extract the stellar radius.
    d = default_data['sy_dist'][0] # Extract the distance.

    return default_data, T, R_star, d

In [ ]:
# Call upon the data-extraction function (get_planet_data), making all assigned parameters available.
default_data, T, R_star, d = get_planet_data(planet)

In [ ]:
# Define the parameters.
"""
Use the astropy irreducible standard units. (Our exception is length, for which we will use an SI prefix in order to fit the CGS system for blackbody radiation equation.)
Length (Distance) has units of centimeters (cm).
Time has units of seconds (s).
Temperature has units of Kelvin (K).
"""
planet_name = default_data['pl_name'][0] # Derive the name of the planet from the extracted data.
host_star_name = default_data['hostname'][0] # Derive the name of the host star from the extracted data.
R_star = default_data['st_rad'][0] # Derive the stellar radius from the extracted data. Use units of stellar radii (R_sun).
d = default_data['sy_dist'][0] # Derive the distance from the star to the detector from the extracted data. Use units of parsec (pc).
T = default_data['st_teff'][0] # Derive the stellar effective temperature from the extracted data. Use units of Kelvin (K).
A = pi * (15 * u.m).to(u.cm)**2 # Represent the telescope area in centimeters squared (cm²).
t = 100 * u.s # Represent the exposure time in seconds (s).

print(f"Planet: {planet_name}")
print(f"Host star: {host_star_name}")
print(f"Radius of star: {R_star}")
print(f"Distance from star to detector: {d}")
print(f"Temperature of star: {T}")
print(f"Area: {A}")
print(f"Time: {t}")

Planet: LTT 1445 A c
Host star: LTT 1445 A
Radius of star: 0.265 Rsun
Distance from star to detector: 6.86929 pc
Temperature of star: 3340.0 K
Area: 7068583.470577034 cm2
Time: 100.0 s


In [ ]:
# Define a function for spectral radiance.
def photon_radiance(lambda_, T, A, t, R_star, d):
    # Photon radiance is a function of wavelength, temperature, area, time, distance, and radiance.
    """
    This function calculates the spectral radiance of a star.
    Input:
    lambda_, the wavelength of the electromagnetic radiation (m).
    T, the stellar effective temperature (K).
    A, the telescope collecting area (m**2).
    t, the exposure time (s).
    R_star, the stellar radius (m).
    d, the distance from the star to the detector (m).
    Output:
    spectral_radiance, the spectral radiance. (W / (m**2 * Hz * sr))
    """
    photon_flux = (2 * c / (lambda_**4)) / (np.exp((h*c) / (lambda_ * k_B * T)) - 1) # Define the equation for spectral radiance.

    scaling_factor = (A * t * (R_star / d)**2) # Define the equation of the scaling factor.

    return photon_flux * scaling_factor

# What do we know about numpy.trapezoid?

*   The Trapezoidal Rule is a method of approximating integration by splitting an area we wish to integrate into trapezoids and summing the areas of each trapezoid.
*  As $n$ increases, $\Delta x$ gets progressively smaller and we sum more and more trapezoids, and the approximation of the integral becomes more accurate.
      *   I use high values of n because it makes the integral more accurate.
*   While numpy.trapezoid is array-based, we are integrating along a parametric curve. To do so, we must define x and y where x is in np.linspace (an array of evenly-spaced points). We must also define $n$, which tells us how precise our approximation is.

In [ ]:
# Define a function for photon count.
def photon_count_method1(planet_name, lambda_min, lambda_max, A, t, n_points=1000): # Elements have feature wavelengths. The range is used for the bound of the integral.
    x = np.linspace(lambda_min, lambda_max, num=n_points) # Define the x variable. Consider integration to be done along the x-y scale.
    y = photon_radiance(x, T, A, t, R_star, d) # Define the y variable. Consider integration to be done along the x-y scale.
    photon_count = np.trapezoid(y, x) # Define photon count as the calculated integral.

    return photon_count

In [ ]:
# Define the feature wavelength ranges for all four biosignture gases. These will define the upper and lower limits of integration.
biosignature_gas_ranges = {"O2": (1.25 * u.um, 1.28 * u.um),
                           "CO2": (4.17 * u.um, 4.51 * u.um),
                           "CH4": (3.41 * u.um, 3.71 * u.um),
                           "H2O": (1.76 * u.um, 1.94 * u.um)}

In [ ]:
# Use a for ... in statement in order to iterate through each biosignature gas and its respective wavelength range.
for gas, (lambda_min, lambda_max) in biosignature_gas_ranges.items():
    # Call upon the photon count function (photon_count_method_1) in order to calculate the photon counts for each biosignature gas for a selected exoplanet.
    photon_count = photon_count_method1(planet, lambda_min, lambda_max, A, t)
    print(f"Photon count for {gas} on {planet}: {(photon_count).decompose()}")

Photon count for O2 on LTT 1445 A c: 12896856753.84266
Photon count for CO2 on LTT 1445 A c: 18129681685.83899
Photon count for CH4 on LTT 1445 A c: 25493791159.193367
Photon count for H2O on LTT 1445 A c: 53224161516.0517
